In [1]:
"""
TimeKAN SPA-HAR — UCI-HAR 분류 (Google Colab 즉시 실행 가능)

[이번 버전 핵심 3가지]
  1. Periodicity Gating   : 반복성 강한 샘플에만 phase loss 강하게 적용
  2. Phase Feature 고도화 : 2D 전체 평균 → 4구간×2D = 8D 시간 구조 반영
  3. Non-Collapse Anchor  : phase head가 상수 벡터로 죽지 않도록 약한 spread 항 추가
"""

# ── Colab 의존성 설치 (로컬 실행 시 주석 처리) ──────────────────────────────
# !pip install scikit-learn -q

import math, os
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from sklearn.metrics import classification_report, confusion_matrix

# ─────────────────────────────────────────────────────────────────────────────
# 1. UCI-HAR 데이터 로더
# ─────────────────────────────────────────────────────────────────────────────
UCI_HAR_URL = (
    "https://archive.ics.uci.edu/ml/machine-learning-databases"
    "/00240/UCI%20HAR%20Dataset.zip"
)
LABEL_NAMES = [
    "WALKING", "WALKING_UPSTAIRS", "WALKING_DOWNSTAIRS",
    "SITTING",  "STANDING",         "LAYING",
]
SAMPLE_RATE = 50   # Hz

def download_ucihar(root="./data"):
    import urllib.request, zipfile
    os.makedirs(root, exist_ok=True)
    dataset_dir = os.path.join(root, "UCI HAR Dataset")
    if not os.path.exists(dataset_dir):
        print("Downloading UCI-HAR dataset...")
        zip_path = os.path.join(root, "ucihar.zip")
        urllib.request.urlretrieve(UCI_HAR_URL, zip_path)
        with zipfile.ZipFile(zip_path, "r") as zf:
            zf.extractall(root)
        print("Download complete.")
    return dataset_dir

def load_raw_signals(base_dir, split):
    """9축 원시 관성 신호 → (N, 9, 128)"""
    names = [
        "body_acc_x",  "body_acc_y",  "body_acc_z",
        "body_gyro_x", "body_gyro_y", "body_gyro_z",
        "total_acc_x", "total_acc_y", "total_acc_z",
    ]
    sigs = [
        np.loadtxt(os.path.join(base_dir, "Inertial Signals", f"{n}_{split}.txt"))
        for n in names
    ]
    return np.stack(sigs, axis=1).astype(np.float32)

def load_labels(base_dir, split):
    return np.loadtxt(os.path.join(base_dir, f"y_{split}.txt")).astype(np.int64) - 1

def normalize_signals(X_train, X_test):
    mean = X_train.mean(axis=(0, 2), keepdims=True)
    std  = X_train.std(axis=(0, 2),  keepdims=True) + 1e-8
    return (X_train - mean) / std, (X_test - mean) / std

class UCIHARDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.from_numpy(X)
        self.y = torch.from_numpy(y)
    def __len__(self):           return len(self.y)
    def __getitem__(self, idx):  return self.X[idx], self.y[idx]


# ─────────────────────────────────────────────────────────────────────────────
# 2. Periodicity Score  ← 변경①의 핵심
#
#  body_acc magnitude (ch 0-2) 의 자기상관(autocorrelation) 최대값으로
#  배치 내 각 샘플의 반복성 신뢰도를 [0, 1] 스케일로 계산
#
#  동작 원리:
#    반복운동(걷기) → 일정 lag에서 AC 피크 높음 → score ≈ 1
#    정적 활동(앉기) → AC 피크 낮음           → score ≈ 0
#
#  활용: score를 cycle loss / phase contrastive 가중치로 사용
#        → "주기성이 있을 때만 phase prior를 강하게 건다"
# ─────────────────────────────────────────────────────────────────────────────
def periodicity_score(X):
    """
    X: (B, 9, T)
    반환: (B,) ∈ [0, 1],  .detach() 포함 (gradient 차단)
    """
    # body_acc 3축 → magnitude (B, T)
    mag = torch.norm(X[:, :3, :], dim=1)
    mag = mag - mag.mean(dim=1, keepdim=True)   # DC 제거

    T    = X.size(2)
    lags = range(2, T // 2)                     # lag 범위: 2 ~ T/2
    ac_list = []
    for lag in lags:
        a  = mag[:, :-lag]
        b  = mag[:, lag:]
        ac = (a * b).mean(dim=1)                # (B,)
        ac_list.append(ac)

    ac    = torch.stack(ac_list, dim=1)          # (B, L)
    score = ac.max(dim=1)[0]                     # (B,) 최대 AC 피크
    score = score.clamp(min=0.0)                 # 음수 제거
    score = score / (score.max().detach() + 1e-8)  # [0,1] 정규화

    return score.detach()                        # gradient 차단 (gate만 역할)


# ─────────────────────────────────────────────────────────────────────────────
# 3. HAR 맞춤형 Adaptive Series Decomposition
#
#  멀티스케일 이동평균 + 채널별 어텐션으로 유효 스케일 학습
#    k=5  → 0.10s : 고주파 (보행 충격)
#    k=15 → 0.30s : 반주기 (swing phase)
#    k=25 → 0.50s : 1주기  (기존 고정값)
#    k=51 → 1.02s : 저주파 (자세 전환)
# ─────────────────────────────────────────────────────────────────────────────
class _FixedAvg(nn.Module):
    def __init__(self, k):
        super().__init__()
        self.k   = k
        self.avg = nn.AvgPool1d(kernel_size=k, stride=1, padding=0)

    def forward(self, x):   # x: (B, C, T)
        pad = (self.k - 1) // 2
        return self.avg(F.pad(x, (pad, pad), mode="replicate"))

class AdaptiveSeriesDecomp(nn.Module):
    """
    Trend(x,t) = Σ_k  w_k(x,t) · AvgPool_k(x),  softmax over k
    Resid      = x - Trend

    scale_attn: depthwise Conv1d (채널 혼합 없이 채널별 스케일 선택)
    """
    def __init__(self, d_model, scales=(5, 15, 25, 51)):
        super().__init__()
        self.S    = len(scales)
        self.avgs = nn.ModuleList([_FixedAvg(k) for k in scales])
        self.scale_attn = nn.Conv1d(
            d_model, d_model * self.S,
            kernel_size=1, groups=d_model, bias=True,
        )

    def forward(self, x):   # x: (B, d_model, T)
        B, C, T = x.shape
        w    = self.scale_attn(x).view(B, C, self.S, T)          # (B, C, S, T)
        w    = torch.softmax(w, dim=2)
        avgs = torch.stack([a(x) for a in self.avgs], dim=2)     # (B, C, S, T)
        trend = (w * avgs).sum(dim=2)                             # (B, C, T)
        return x - trend, trend


# ─────────────────────────────────────────────────────────────────────────────
# 4. ChebyshevLinear
# ─────────────────────────────────────────────────────────────────────────────
class ChebyshevLinear(nn.Module):
    def __init__(self, in_features, out_features, degree):
        super().__init__()
        self.degree = degree
        self.weight = nn.Parameter(
            torch.empty(degree + 1, in_features, out_features)
        )
        nn.init.kaiming_uniform_(self.weight, a=math.sqrt(5))

    def forward(self, x):   # x: (B, T, C)
        x  = x.clamp(-1.0, 1.0)    # 체비셰프 직교 구간 강제
        Tl = [torch.ones_like(x), x]
        for i in range(2, self.degree + 1):
            Tl.append(2 * x * Tl[i - 1] - Tl[i - 2])
        return sum(torch.matmul(Tl[i], self.weight[i])
                   for i in range(self.degree + 1))


# ─────────────────────────────────────────────────────────────────────────────
# 5. TimeKANBlock / TimeKANEncoder
# ─────────────────────────────────────────────────────────────────────────────
class TimeKANBlock(nn.Module):
    def __init__(self, d_model, scales=(5, 15, 25, 51)):
        super().__init__()
        self.decomp    = AdaptiveSeriesDecomp(d_model, scales)
        self.dw_conv   = nn.Conv1d(d_model, d_model, kernel_size=3,
                                   padding=1, groups=d_model)
        self.kan_res   = ChebyshevLinear(d_model, d_model, degree=3)
        self.kan_trend = ChebyshevLinear(d_model, d_model, degree=1)
        self.norm      = nn.LayerNorm(d_model)
        self.drop      = nn.Dropout(0.1)

    def forward(self, x):   # x: (B, T, d_model)
        residual      = x
        xc            = self.dw_conv(x.transpose(1, 2))        # (B, d_model, T)
        r_t, tr_t     = self.decomp(xc)
        res           = self.kan_res(r_t.transpose(1, 2))       # (B, T, d_model)
        trend         = self.kan_trend(tr_t.transpose(1, 2))
        return self.norm(self.drop(res + trend) + residual)

class TimeKANEncoder(nn.Module):
    def __init__(self, in_channels=9, d_model=128, num_blocks=3,
                 scales=(5, 15, 25, 51)):
        super().__init__()
        self.proj   = nn.Linear(in_channels, d_model)
        self.blocks = nn.ModuleList(
            [TimeKANBlock(d_model, scales) for _ in range(num_blocks)]
        )
        self.norm   = nn.LayerNorm(d_model)

    def forward(self, x):   # x: (B, 9, T)
        x = self.proj(x.transpose(1, 2))    # (B, T, d_model)
        for blk in self.blocks:
            x = blk(x)
        return self.norm(x)                  # (B, T, d_model)


# ─────────────────────────────────────────────────────────────────────────────
# 6. MotionPhaseHead  (단위원 위상 + 고도화된 특징)
# ─────────────────────────────────────────────────────────────────────────────
class MotionPhaseHead(nn.Module):
    """
    출력: (B, T, 2) — 단위원 (cosφ, sinφ)
    위상 속도 Δφ(t) = atan2(sin(Δφ), cos(Δφ)) ∈ [-π, π]  (랩핑 없음)
    """
    def __init__(self, d_model):
        super().__init__()
        self.fc = nn.Sequential(
            nn.Linear(d_model, d_model // 2), nn.GELU(),
            nn.Linear(d_model // 2, 2),
        )

    def forward(self, H):                       # H: (B, T, d_model)
        return F.normalize(self.fc(H), dim=-1)  # (B, T, 2) 단위원

    # ── 위상 속도 ─────────────────────────────────────────────────────────────
    @staticmethod
    def phase_velocity(phase_cs):
        """
        Δφ(t) ∈ [-π, π]  — 단위원 외적/내적으로 랩핑 없이 계산
        반환: (B, T-1)
        """
        c,  s  = phase_cs[:, :-1, 0], phase_cs[:, :-1, 1]
        c1, s1 = phase_cs[:, 1:,  0], phase_cs[:, 1:,  1]
        cross  = c * s1 - s * c1    # sin(Δφ)
        dot    = c * c1 + s * s1    # cos(Δφ)
        return torch.atan2(cross, dot)   # (B, T-1)

    # ── 변경②: Phase Feature 고도화  2D → 8D ────────────────────────────────
    @staticmethod
    def phase_velocity_feat(phase_cs, n_segments=4):
        """
        [기존] 전체 시퀀스 평균 → (B, 2)  — 시간 구조 전혀 반영 안 됨
        [개선] 4구간 분할 × (sin평균, cos평균) → (B, 8)  — 위상 dynamics 반영

        n_segments=4 기준:
          seg0: 0    ~ T/4   (초반 보행 패턴)
          seg1: T/4  ~ T/2
          seg2: T/2  ~ 3T/4
          seg3: 3T/4 ~ T-1   (후반 보행 패턴)
        """
        dv    = MotionPhaseHead.phase_velocity(phase_cs)   # (B, T-1)
        B, Tm1 = dv.shape
        seg_len = Tm1 // n_segments
        feats = []
        for i in range(n_segments):
            s   = i * seg_len
            e   = Tm1 if i == n_segments - 1 else (i + 1) * seg_len
            seg = dv[:, s:e]                               # (B, seg_len)
            feats.append(torch.sin(seg).mean(dim=1))       # (B,)  sin 평균
            feats.append(torch.cos(seg).mean(dim=1))       # (B,)  cos 평균
        feat = torch.stack(feats, dim=-1)                  # (B, 2*n_segments)
        return F.normalize(feat, dim=-1)                   # L2 정규화


# ─────────────────────────────────────────────────────────────────────────────
# 7. ProjectionHead
# ─────────────────────────────────────────────────────────────────────────────
class ProjectionHead(nn.Module):
    def __init__(self, d_model, proj_dim=64):
        super().__init__()
        self.fc = nn.Sequential(
            nn.Linear(d_model, d_model), nn.GELU(),
            nn.Linear(d_model, proj_dim),
        )
    def forward(self, H):
        return F.normalize(self.fc(H), dim=-1)   # (B, T, proj_dim)


# ─────────────────────────────────────────────────────────────────────────────
# 8. SPAHARModel
# ─────────────────────────────────────────────────────────────────────────────
class SPAHARModel(nn.Module):
    def __init__(self, in_channels=9, d_model=128, n_classes=6,
                 proj_dim=64, scales=(5, 15, 25, 51)):
        super().__init__()
        self.encoder    = TimeKANEncoder(in_channels, d_model,
                                         num_blocks=3, scales=scales)
        self.classifier = nn.Sequential(
            nn.Linear(d_model, 128), nn.GELU(), nn.Dropout(0.3),
            nn.Linear(128, n_classes),
        )
        self.phase_head = MotionPhaseHead(d_model)
        self.proj_head  = ProjectionHead(d_model, proj_dim)

    def forward(self, x):
        H      = self.encoder(x)                # (B, T, d_model)
        logits = self.classifier(H.mean(1))     # (B, n_classes)
        phase  = self.phase_head(H)             # (B, T, 2)  단위원
        z      = self.proj_head(H)              # (B, T, proj_dim)
        return logits, H, phase, z


# ─────────────────────────────────────────────────────────────────────────────
# 9. 손실 함수
# ─────────────────────────────────────────────────────────────────────────────

# ── 변경①: Periodicity-Gated Cycle-Consistency Loss ──────────────────────────
def cycle_consistency_loss(phase_cs, X):
    """
    [기존] L_cyc = E[ Var_t(Δφ) ]  — 모든 샘플에 동일 가중치
    [개선] L_cyc = E[ score(x) · Var_t(Δφ) ]  — 반복성 강한 샘플에 집중

    score(x) ∈ [0,1]: body_acc 자기상관 최대 피크
      → 걷기 ≈ 1,  앉기·서기 ≈ 0
    → "주기성이 있을 때만 phase variance를 최소화하라"
    """
    score   = periodicity_score(X)                              # (B,) detach됨
    dv      = MotionPhaseHead.phase_velocity(phase_cs)          # (B, T-1)
    cyc_each = dv.var(dim=1)                                    # (B,)
    return (score * cyc_each).mean()

# ── 변경①: Periodicity-Gated Phase-Aware Contrastive Loss ────────────────────
def phase_aware_contrastive_loss(phase_cs, labels, X, temperature=0.07):
    """
    [기존] 모든 샘플을 동등하게 대조 학습
    [개선] 반복성 score로 각 샘플 쌍의 positive/negative 가중치 조정

    positive 쌍 가중치  = score_i × score_j  (둘 다 반복적일 때만 강하게 pull)
    negative 쌍 가중치  = 1 (변경 없음; push는 항상 유효)

    feat: (B, 8)  ← 8D phase velocity feature (변경②)
    """
    feat   = MotionPhaseHead.phase_velocity_feat(phase_cs)  # (B, 8)
    sim    = torch.mm(feat, feat.T) / temperature            # (B, B)
    score  = periodicity_score(X)                            # (B,) detach

    # Positive mask: 같은 클래스, 자기 자신 제외
    lbl      = labels.unsqueeze(1)
    pos_mask = (lbl == lbl.T).float()
    eye      = torch.eye(len(labels), device=labels.device)
    pos_mask = pos_mask - eye

    # 반복성 가중치: pos_weight[i,j] = score_i × score_j
    score_outer = score.unsqueeze(1) * score.unsqueeze(0)   # (B, B)
    pos_mask    = pos_mask * score_outer                     # 가중된 positive

    # Numerically stable log-softmax
    sim_max  = sim.detach().max(dim=1, keepdim=True)[0]
    exp_sim  = torch.exp(sim - sim_max)
    denom    = (exp_sim * (1 - eye)).sum(dim=1, keepdim=True).clamp(min=1e-8)
    log_prob = (sim - sim_max) - torch.log(denom)

    n_pos = pos_mask.sum(dim=1).clamp(min=1)
    loss  = -(pos_mask * log_prob).sum(dim=1) / n_pos
    return loss.mean()

# ── 변경③: Phase Non-Collapse Anchor ─────────────────────────────────────────
def phase_noncollapse_loss(phase_cs):
    """
    phase head가 상수 벡터(trivial collapse)로 수렴하는 것을 방지
    L_nc = E[ exp(-Var_t(phase_cs)) ]

    - 시간축 분산이 크면 → exp(-큰값) ≈ 0  (loss 작음, 패널티 없음)
    - 시간축 분산이 0에 가까우면 → exp(-0) = 1  (loss 최대, 강한 패널티)
    - lambda_nc = 0.005~0.01 수준의 약한 가중치로 사용
      (목적: 성능 향상이 아니라 phase branch 생존 보장)
    """
    var_t = phase_cs.var(dim=1).mean(dim=1)   # (B,)  시간축 분산 평균
    return torch.exp(-var_t).mean()


# ─────────────────────────────────────────────────────────────────────────────
# 10. 학습 / 평가 루프
# ─────────────────────────────────────────────────────────────────────────────
def train_epoch(model, loader, optimizer, device,
                lambda_cyc=0.05, lambda_pac=0.10, lambda_nc=0.01):
    """
    전체 손실:
      L = L_CE
        + lambda_cyc · L_cycle(gated)          ← 변경①
        + lambda_pac · L_phase_contrastive(gated)  ← 변경①②
        + lambda_nc  · L_noncollapse           ← 변경③

    권장 탐색 범위:
      lambda_cyc : [0.01, 0.05, 0.10]
      lambda_pac : [0.05, 0.10, 0.20]
      lambda_nc  : [0.005, 0.01]  (성능보다 안정성 목적; 너무 크면 안 됨)
    """
    model.train()
    total_loss, correct, total = 0.0, 0, 0

    for X, y in loader:
        X, y = X.to(device), y.to(device)
        logits, H, phase, z = model(X)

        ce   = F.cross_entropy(logits, y)
        cyc  = cycle_consistency_loss(phase, X)          # gated ①
        pac  = phase_aware_contrastive_loss(phase, y, X) # gated ①②
        nc   = phase_noncollapse_loss(phase)             # anchor ③
        loss = ce + lambda_cyc * cyc + lambda_pac * pac + lambda_nc * nc

        optimizer.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        total_loss += loss.item() * len(y)
        correct    += (logits.argmax(1) == y).sum().item()
        total      += len(y)

    return total_loss / total, correct / total


@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()
    preds_all, labels_all = [], []
    for X, y in loader:
        logits, *_ = model(X.to(device))
        preds_all.append(logits.argmax(1).cpu())
        labels_all.append(y)
    preds  = torch.cat(preds_all).numpy()
    labels = torch.cat(labels_all).numpy()
    return (preds == labels).mean(), preds, labels


# ─────────────────────────────────────────────────────────────────────────────
# 11. 메인 실행
# ─────────────────────────────────────────────────────────────────────────────
def main():
    # ── 하이퍼파라미터 ─────────────────────────────────────────────────────────
    DATA_ROOT  = "/content/drive/MyDrive/Colab Notebooks/UCI-HAR/UCI-HAR"
    BATCH_SIZE = 256
    EPOCHS     = 50
    LR         = 3e-4
    D_MODEL    = 128
    PROJ_DIM   = 64
    N_CLASSES  = 6

    # Adaptive Decomp 스케일 (50Hz 기준 0.1/0.3/0.5/1.0s)
    # ablation: (5, 25) / (5, 15, 51) 등으로 변경해 비교 가능
    SCALES = (5, 15, 25, 51)

    # Phase 보조 손실 가중치
    LAMBDA_CYC = 0.05   # periodicity-gated cycle consistency
    LAMBDA_PAC = 0.10   # periodicity-gated phase contrastive
    LAMBDA_NC  = 0.01   # non-collapse anchor (약하게 고정)

    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Device      : {DEVICE}")
    print(f"Scales      : {SCALES} samples "
          f"≈ {[round(s / SAMPLE_RATE, 2) for s in SCALES]}s @ {SAMPLE_RATE}Hz")
    print(f"Loss weights: cyc={LAMBDA_CYC}  pac={LAMBDA_PAC}  nc={LAMBDA_NC}")

    # ── 데이터 ────────────────────────────────────────────────────────────────
    dataset_dir = download_ucihar(DATA_ROOT)
    X_train = load_raw_signals(os.path.join(dataset_dir, "train"), "train")
    y_train = load_labels(os.path.join(dataset_dir, "train"), "train")
    X_test  = load_raw_signals(os.path.join(dataset_dir, "test"),  "test")
    y_test  = load_labels(os.path.join(dataset_dir, "test"),  "test")
    X_train, X_test = normalize_signals(X_train, X_test)

    train_dl = DataLoader(UCIHARDataset(X_train, y_train),
                          batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=2, pin_memory=True)
    test_dl  = DataLoader(UCIHARDataset(X_test, y_test),
                          batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=2, pin_memory=True)
    print(f"Train       : {len(y_train)} | Test : {len(y_test)}")

    # ── 모델 ──────────────────────────────────────────────────────────────────
    model     = SPAHARModel(in_channels=9, d_model=D_MODEL,
                            n_classes=N_CLASSES, proj_dim=PROJ_DIM,
                            scales=SCALES).to(DEVICE)
    optimizer = AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
    scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)

    n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Parameters  : {n_params:,}")
    print("─" * 65)

    # ── 학습 루프 ─────────────────────────────────────────────────────────────
    best_acc = 0.0
    for epoch in range(1, EPOCHS + 1):
        tr_loss, tr_acc = train_epoch(
            model, train_dl, optimizer, DEVICE,
            LAMBDA_CYC, LAMBDA_PAC, LAMBDA_NC,
        )
        val_acc, _, _ = evaluate(model, test_dl, DEVICE)
        scheduler.step()

        is_best = val_acc > best_acc
        if is_best:
            best_acc = val_acc
            torch.save(model.state_dict(), "best_model.pth")

        if epoch % 5 == 0 or epoch == 1:
            tag = " ◀ best" if is_best else ""
            print(f"[Epoch {epoch:3d}/{EPOCHS}]  "
                  f"loss={tr_loss:.4f}  "
                  f"train={tr_acc:.4f}  "
                  f"test={val_acc:.4f}{tag}")

    # ── 최종 평가 ─────────────────────────────────────────────────────────────
    print("─" * 65)
    print(f"Best test accuracy : {best_acc:.4f}")
    model.load_state_dict(torch.load("best_model.pth", map_location=DEVICE))
    _, preds, labels = evaluate(model, test_dl, DEVICE)

    print("\n── Classification Report ─────────────────────────────────────")
    print(classification_report(labels, preds,
                                target_names=LABEL_NAMES, digits=4))

    print("── Confusion Matrix ──────────────────────────────────────────")
    cm  = confusion_matrix(labels, preds)
    hdr = "         " + "  ".join(f"{n[:5]:>5}" for n in LABEL_NAMES)
    print(hdr)
    for i, row in enumerate(cm):
        print(f"{LABEL_NAMES[i][:8]:>8} " +
              "  ".join(f"{v:7d}" for v in row))


if __name__ == "__main__":
    main()

Device      : cuda
Scales      : (5, 15, 25, 51) samples ≈ [0.1, 0.3, 0.5, 1.02]s @ 50Hz
Loss weights: cyc=0.05  pac=0.1  nc=0.01
Train       : 7352 | Test : 2947
Parameters  : 352,264
─────────────────────────────────────────────────────────────────
[Epoch   1/50]  loss=1.4554  train=0.5895  test=0.7275 ◀ best
[Epoch   5/50]  loss=0.4173  train=0.9331  test=0.8999
[Epoch  10/50]  loss=0.3512  train=0.9455  test=0.9135
[Epoch  15/50]  loss=0.3316  train=0.9487  test=0.9274 ◀ best
[Epoch  20/50]  loss=0.3301  train=0.9480  test=0.9192
[Epoch  25/50]  loss=0.3069  train=0.9582  test=0.9192
[Epoch  30/50]  loss=0.3047  train=0.9612  test=0.9342
[Epoch  35/50]  loss=0.2782  train=0.9645  test=0.9213
[Epoch  40/50]  loss=0.2690  train=0.9675  test=0.9318
[Epoch  45/50]  loss=0.2645  train=0.9672  test=0.9281
[Epoch  50/50]  loss=0.2615  train=0.9686  test=0.9281
─────────────────────────────────────────────────────────────────
Best test accuracy : 0.9393

── Classification Report ──────────